# Model 2 — baseline + keyword aroma features

Tests whether the Sprint-5 heuristic keyword features improve price
prediction over the plain baseline XGBoost model.

Both feature tables are precomputed and loaded here — no encoding or
keyword scoring happens in this notebook:

- `wine_features.parquet`  (from `04_feature_engineering.ipynb`)
- `wine_keywords.parquet`  (from `05_nlp_keywords.ipynb`)

joined on `wine_id`. **Model 1** uses the base features; **model 2** adds
the 8 `kw_<aroma>` density columns. Everything else (split, trim, untuned
`XGBRegressor`) is identical, so any metric difference is attributable to
the aroma features.

In [ ]:
import pandas as pd
import itables
from itables import show
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import xgboost as xgb

itables.options.columnDefs = [{"className": "dt-left", "targets": "_all"}]

FEATURES_PATH = r"..\..\.data\wine_features.parquet"
KEYWORDS_PATH = r"..\..\.data\wine_keywords.parquet"

features = pd.read_parquet(FEATURES_PATH)
keywords = pd.read_parquet(KEYWORDS_PATH)
print(f"features: {features.shape}  |  keywords: {keywords.shape}")

kw_features = [c for c in keywords.columns if c.startswith("kw_") and not c.endswith("_count")]
df = features.merge(keywords[["wine_id"] + kw_features], on="wine_id", how="left")
assert len(df) == len(features), "merge changed row count — wine_id not unique?"
print(f"merged: {df.shape}  |  aroma features: {kw_features}")

## Train & compare

Both models train on the *same* rows and split; model 2 only adds the
`kw_*` columns.

In [ ]:
base_features = [
    "rating", "alcohol", "bottle_size", "vintage", "case_production",
    "country_ord", "wine_type_ord", "state_ord", "company_ord",
    "appellation_ord", "varietal_label_ord",
]
target = "retail"

model_df = df[base_features + kw_features + [target]].dropna(subset=[target])
low, high = model_df[target].quantile([0.02, 0.90])
model_df = model_df[model_df[target].between(low, high)]
print(f"Retail range after trimming: ${low:.2f} - ${high:.2f}  ({len(model_df):,} rows)")


def train_eval(feature_list):
    X, y = model_df[feature_list], model_df[target]
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    model = xgb.XGBRegressor(random_state=42)
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    return model, {
        "n_features": len(feature_list),
        "RMSE": mean_squared_error(y_test, pred) ** 0.5,
        "MAE":  mean_absolute_error(y_test, pred),
        "R2":   r2_score(y_test, pred),
    }


model1, m1 = train_eval(base_features)
model2, m2 = train_eval(base_features + kw_features)

results = pd.DataFrame([{"model": "1: baseline", **m1},
                        {"model": "2: baseline + kw", **m2}])
results.loc["delta"] = ["delta vs baseline", "",
                        m2["RMSE"] - m1["RMSE"], m2["MAE"] - m1["MAE"], m2["R2"] - m1["R2"]]
results.round(4)

## Feature importance — model 2

Where the aroma features land relative to the structured ones.

In [ ]:
imp = (
    pd.DataFrame({"feature": base_features + kw_features, "importance": model2.feature_importances_})
    .assign(is_keyword=lambda d: d["feature"].str.startswith("kw_"))
    .sort_values("importance", ascending=False)
    .reset_index(drop=True)
    .assign(importance=lambda d: d["importance"].round(3))
)
show(imp)

## Conclusion

**The heuristic keyword aroma features do not improve the price model — they
slightly degrade it** (test R² 0.6404 → ~0.627, RMSE and MAE both up). The
8 `kw_*` features rank at the bottom of importance.

Why: the aroma keywords mostly encode wine *style*, which `wine_type` and
`varietal_label` already capture. As extra weak, noisy inputs they give an
untuned XGBoost room to overfit the train set without generalising.

This keeps model 1 as the model to beat. Next experiments before concluding
text is useless for price:
- **Sentence embeddings** (Sprint 6) — far richer text signal than keyword counts.
- Regularise model 2 (lower `max_depth`, add `reg_lambda`).
- Keep the keyword features for the **report/BI** (interpretable) rather than
  the price model.